# Part 2: Reply routing decision model

Fine-tunes `LiquidAI/LFM3-1.7B` with Unsloth's Clef decision head on the repository's synthetic reply dataset. Companion to the tutorial on raoofaltaher.com.

In [1]:
import os
import time
from pathlib import Path

if Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)

HF_USERNAME = "raoofaltaher"
PUBLISH = True
LOAD_IN_4BIT = True
TIMINGS = {}

print("Working directory:", Path.cwd())

Working directory: /home/dev/work/lead-decision-models


## Environment

In [2]:
import platform
import torch

print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA runtime:", torch.version.cuda)
else:
    raise RuntimeError("A supported GPU runtime is required for this training run.")

Python: 3.13.18
PyTorch: 2.14.1+cu132
CUDA available: True
GPU: NVIDIA GeForce RTX 5090 Laptop GPU
CUDA runtime: 13.2


In [3]:
import json
from collections import Counter

import numpy as np
import pandas as pd
from datasets import Dataset
from sklearn.model_selection import StratifiedGroupKFold
from unsloth import DecisionTrainer, FastDecisionModel, is_bfloat16_supported
from transformers import TrainingArguments

MODEL_ID = "LiquidAI/LFM3-1.7B"
MODEL_REVISION = "ac2586a34f1a1c03329dbdcdefabd8b6478ddaa7"
print("Model:", MODEL_ID, "pinned revision:", MODEL_REVISION)


def to_decision_dataset(frame, builder):
    rows = [
        builder(row)
        for row in frame.to_dict(orient="records")
    ]
    return Dataset.from_list(rows)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


W1103 15:35:14.429000 4486 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


W1103 15:35:14.459000 4486 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


/home/dev/work/lead-decision-models/unsloth_env/lib/python3.13/site-packages/transformers/configuration_utils.py:723: FutureWarning: `rope_scaling` is deprecated and will be removed in a future version. Set `rope_parameters` instead, with `rope_type` and `rope_theta` included.
  warnings.warn(


🦥 Unsloth Zoo will now patch everything to make training faster!


Model: LiquidAI/LFM3-1.7B pinned revision: ac2586a34f1a1c03329dbdcdefabd8b6478ddaa7


## Reply dataset

In [4]:
REPLY_INTENTS = [
    "interested",
    "meeting_requested",
    "more_information",
    "referral",
    "not_now",
    "objection",
    "wrong_person",
    "out_of_office",
    "not_interested",
    "opt_out",
    "ambiguous",
]

reply_df = pd.read_csv("data/replies/reply_examples.csv")

required_reply_columns = {
    "reply_id",
    "scenario_group",
    "outreach_context",
    "reply_text",
    "reply_intent",
    "is_synthetic",
}

missing_reply_columns = required_reply_columns - set(reply_df.columns)
if missing_reply_columns:
    raise ValueError(f"Missing reply columns: {sorted(missing_reply_columns)}")

reply_df["is_synthetic"] = (
    reply_df["is_synthetic"]
    .astype(str)
    .str.strip()
    .str.lower()
    .eq("true")
)

reply_df["reply_intent"] = (
    reply_df["reply_intent"]
    .astype(str)
    .str.strip()
)

if not reply_df["is_synthetic"].all():
    raise ValueError("This experiment accepts only explicitly synthetic reply rows.")

if reply_df["reply_id"].duplicated().any():
    raise ValueError("reply_id values must be unique.")

normalized_reply_text = (
    reply_df["reply_text"]
    .astype(str)
    .str.strip()
    .str.casefold()
)

if normalized_reply_text.duplicated().any():
    raise ValueError(
        "Exact duplicate reply text exists. Deduplicate or review it before splitting."
    )

required_text_columns = [
    "reply_id",
    "scenario_group",
    "outreach_context",
    "reply_text",
]

if reply_df[required_text_columns].isna().any().any():
    raise ValueError("Reply IDs, groups, context and text cannot be null.")

if (
    reply_df[required_text_columns]
    .astype(str)
    .apply(lambda column: column.str.strip().eq("").any())
    .any()
):
    raise ValueError("Reply IDs, groups, context and text cannot be blank.")

if not set(reply_df["reply_intent"]).issubset(set(REPLY_INTENTS)):
    raise ValueError("The dataset contains an unknown reply_intent.")

if reply_df.groupby("scenario_group")["reply_intent"].nunique().gt(1).any():
    raise ValueError("A scenario_group cannot contain multiple reply_intent labels.")

if len(reply_df) < 500:
    raise ValueError("The reply dataset must hold at least 500 rows.")

class_counts = reply_df["reply_intent"].value_counts()
print(class_counts)

missing_classes = set(REPLY_INTENTS) - set(class_counts.index)
if missing_classes:
    raise ValueError(f"Missing reply classes: {sorted(missing_classes)}")

if class_counts.min() < 20:
    raise ValueError("Add more examples to the under-represented classes.")

group_counts = reply_df.groupby("reply_intent")["scenario_group"].nunique()
if group_counts.min() < 10:
    raise ValueError(
        "Each reply intent needs at least ten independent scenario groups "
        "for the five-fold grouped split."
    )

from hashlib import sha256
from pathlib import Path

reply_dataset_hash = sha256(
    Path("data/replies/reply_examples.csv").read_bytes()
).hexdigest()
published_hash = Path("data/replies/reply_examples.sha256").read_text(encoding="utf-8").split()[0]

if reply_dataset_hash != published_hash:
    raise ValueError("reply_examples.csv does not match its published SHA-256.")

print("Reply dataset SHA-256 verified:", reply_dataset_hash)

reply_intent
interested           61
referral             61
objection            61
wrong_person         61
ambiguous            60
not_interested       56
more_information     53
meeting_requested    51
not_now              51
out_of_office        51
opt_out              51
Name: count, dtype: int64
Reply dataset SHA-256 verified: c57a26d99d05b29b3ce63c5e627d2a79dd03ed78d3bd35ce1afb1f07fca72439


## Decision records and grouped splits

In [5]:
REPLY_QUESTIONS = {
    "reply_intent": {
        "type": "choice",
        "instructions": (
            "Classify the recipient's reply using the defined intent taxonomy. "
            "Do not infer interest that the recipient did not express."
        ),
        "criteria": {
            "interested": "The recipient expresses interest without requesting a meeting.",
            "meeting_requested": "The recipient asks to schedule or discuss a meeting.",
            "more_information": "The recipient requests details, pricing or documentation.",
            "referral": "The recipient points to another relevant person.",
            "not_now": "The recipient asks to revisit the topic later.",
            "objection": "The recipient raises a concern or reason not to proceed.",
            "wrong_person": "The recipient says they are not responsible for this topic.",
            "out_of_office": "The recipient indicates an absence or return date.",
            "not_interested": "The recipient declines further engagement.",
            "opt_out": "The recipient asks to stop marketing or unsubscribe.",
            "ambiguous": "The reply does not support a reliable intent classification.",
        },
    },
    "follow_up_recommended": {
        "type": "noul",
        "instructions": (
            "Does this reply call for an appropriate next step? "
            "A next step may be a handoff, requested information, a pause "
            "or human review. It does not mean an automatic marketing email."
        ),
    },
}

NO_FOLLOW_UP_INTENTS = {
    "not_interested",
    "opt_out",
}

def make_reply_record(row):
    intent = row["reply_intent"]

    state = {
        "task": "B2B sales reply assessment",
        "outreach_context": row["outreach_context"],
        "recipient_reply": row["reply_text"],
    }

    return {
        "state": json.dumps(state, sort_keys=True, ensure_ascii=False),
        "questions": REPLY_QUESTIONS,
        "gold": {
            "reply_intent": intent,
            "follow_up_recommended": intent not in NO_FOLLOW_UP_INTENTS,
        },
    }

In [6]:
reply_splitter = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=3407,
)

reply_folds = list(
    reply_splitter.split(
        reply_df,
        y=reply_df["reply_intent"],
        groups=reply_df["scenario_group"],
    )
)

reply_test_indices = reply_folds[0][1]
reply_validation_indices = reply_folds[1][1]
reply_train_indices = np.concatenate(
    [reply_folds[2][1], reply_folds[3][1], reply_folds[4][1]]
)

reply_train_df = reply_df.iloc[reply_train_indices].reset_index(drop=True)
reply_validation_df = reply_df.iloc[reply_validation_indices].reset_index(drop=True)
reply_test_df = reply_df.iloc[reply_test_indices].reset_index(drop=True)

def scenario_groups(frame):
    return set(frame["scenario_group"])

assert scenario_groups(reply_train_df).isdisjoint(scenario_groups(reply_validation_df))
assert scenario_groups(reply_train_df).isdisjoint(scenario_groups(reply_test_df))
assert scenario_groups(reply_validation_df).isdisjoint(scenario_groups(reply_test_df))

for name, frame in [
    ("train", reply_train_df),
    ("validation", reply_validation_df),
    ("test", reply_test_df),
]:
    missing = set(REPLY_INTENTS) - set(frame["reply_intent"].unique())
    if missing:
        raise ValueError(
            f"{name} split is missing reply classes: {sorted(missing)}. "
            "Add independent scenario groups or revise the split."
        )

    print(name, len(frame))
    print(frame["reply_intent"].value_counts())

train 369
reply_intent
wrong_person         37
interested           36
referral             36
objection            36
ambiguous            36
not_interested       34
more_information     33
opt_out              31
meeting_requested    30
not_now              30
out_of_office        30
Name: count, dtype: int64
validation 124
reply_intent
objection            13
interested           12
referral             12
wrong_person         12
ambiguous            12
not_now              11
out_of_office        11
not_interested       11
meeting_requested    10
more_information     10
opt_out              10
Name: count, dtype: int64
test 124
reply_intent
interested           13
referral             13
objection            12
wrong_person         12
ambiguous            12
meeting_requested    11
not_interested       11
more_information     10
not_now              10
out_of_office        10
opt_out              10
Name: count, dtype: int64


In [7]:
reply_calibration_splitter = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=3408,
)

reply_fit_indices, reply_calibration_indices = next(
    reply_calibration_splitter.split(
        reply_train_df,
        y=reply_train_df["reply_intent"],
        groups=reply_train_df["scenario_group"],
    )
)

reply_fit_df = reply_train_df.iloc[reply_fit_indices].reset_index(drop=True)
reply_calibration_df = reply_train_df.iloc[reply_calibration_indices].reset_index(drop=True)

assert scenario_groups(reply_fit_df).isdisjoint(
    scenario_groups(reply_calibration_df)
)

# The approximate overall proportions are 48% fit, 12% calibration,
# 20% validation and 20% test; grouped row counts will vary.

for name, frame in [
    ("optimizer training", reply_fit_df),
    ("calibration", reply_calibration_df),
]:
    missing = set(REPLY_INTENTS) - set(frame["reply_intent"].unique())
    if missing:
        raise ValueError(
            f"{name} split is missing reply classes: {sorted(missing)}. "
            "Add independent scenario groups or revise the split."
        )

    print(name, len(frame))

optimizer training 296
calibration 73


## Text baseline and model

In [8]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, f1_score
from sklearn.pipeline import Pipeline

reply_text_baseline = Pipeline(
    steps=[
        (
            "tfidf",
            TfidfVectorizer(
                ngram_range=(1, 2),
                min_df=2,
                max_features=50000,
                sublinear_tf=True,
            ),
        ),
        (
            "classifier",
            LogisticRegression(
                max_iter=2000,
                class_weight="balanced",
            ),
        ),
    ]
)

In [9]:
reply_model, reply_tokenizer = FastDecisionModel.from_pretrained(
    model_name=MODEL_ID,
    revision=MODEL_REVISION,
    trust_remote_code=True,
    max_seq_length=2048,
    load_in_4bit=LOAD_IN_4BIT,
)

reply_model = FastDecisionModel.get_peft_model(
    reply_model,
    r=16,
    lora_alpha=16,
    lora_dropout=0.0,
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

reply_train_raw = to_decision_dataset(reply_fit_df, make_reply_record)
reply_calibration_raw = to_decision_dataset(reply_calibration_df, make_reply_record)
reply_validation_raw = to_decision_dataset(reply_validation_df, make_reply_record)
reply_test_raw = to_decision_dataset(reply_test_df, make_reply_record)

reply_train_items_all, reply_train_report = FastDecisionModel.build_dataset(
    reply_train_raw,
    reply_tokenizer,
    reply_model,
)

reply_calibration_items, reply_calibration_report = FastDecisionModel.build_dataset(
    reply_calibration_raw,
    reply_tokenizer,
    reply_model,
)

reply_validation_items, reply_validation_report = FastDecisionModel.build_dataset(
    reply_validation_raw,
    reply_tokenizer,
    reply_model,
)

reply_test_items, reply_test_report = FastDecisionModel.build_dataset(
    reply_test_raw,
    reply_tokenizer,
    reply_model,
)

for name, report in [
    ("train", reply_train_report),
    ("calibration", reply_calibration_report),
    ("validation", reply_validation_report),
    ("test", reply_test_report),
]:
    print(name, report)
    if report["skipped"]:
        raise ValueError(
            f"{name} reply dataset contains skipped decisions: {report['reason']}"
        )

Unsloth: WARNING `trust_remote_code` is True.
Are you certain you want to do remote code execution?
==((====))==  Unsloth 2026.10.11: Fast Lfm3 patching. Transformers: 5.22.1.
   \\   /|    NVIDIA GeForce RTX 5090 Laptop GPU. Num GPUs = 1. Max memory: 23.863 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.14.1+cu132. CUDA: 12.0. CUDA Toolkit: 13.2. Triton: 3.8.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/286 [00:00<?, ?it/s]

Unsloth: Repaired remote modeling code so it trains: Lfm3ForCausalLM.forward.


Unsloth: compiling each transformer block as one region (Lfm3DecoderLayer).


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 7 files:   0%|          | 0/7 [00:00<?, ?it/s]

train {'total': 592, 'skipped': 0, 'reason': None, 'truncated': 0}
calibration {'total': 146, 'skipped': 0, 'reason': None, 'truncated': 0}
validation {'total': 248, 'skipped': 0, 'reason': None, 'truncated': 0}
test {'total': 248, 'skipped': 0, 'reason': None, 'truncated': 0}


In [10]:
reply_train_items = reply_train_items_all

print("Reply optimizer-training rows:", len(reply_train_items))
print("Reply calibration rows:", len(reply_calibration_items))

reply_calibration_counts = Counter(
    question_type
    for item in reply_calibration_items
    for question_type in item["qtypes"]
)

if (
    len(reply_calibration_counts) != len(REPLY_QUESTIONS)
    or min(reply_calibration_counts.values()) < 10
):
    raise ValueError(
        "Provide at least ten calibration decisions for both reply question types. "
        f"Observed counts: {dict(reply_calibration_counts)}"
    )

# Fit the classical baseline on exactly the same scenario groups as Unsloth.
reply_text_baseline.fit(
    reply_fit_df["reply_text"],
    reply_fit_df["reply_intent"],
)

reply_validation_text_predictions = reply_text_baseline.predict(
    reply_validation_df["reply_text"]
)

print(
    classification_report(
        reply_validation_df["reply_intent"],
        reply_validation_text_predictions,
        labels=REPLY_INTENTS,
        zero_division=0,
    )
)

print(
    "TF-IDF validation macro-F1:",
    f1_score(
        reply_validation_df["reply_intent"],
        reply_validation_text_predictions,
        labels=REPLY_INTENTS,
        average="macro",
        zero_division=0,
    ),
)

reply_baseline = FastDecisionModel.evaluate(
    reply_model,
    reply_tokenizer,
    reply_validation_items,
)

print("Unfine-tuned reply baseline:", reply_baseline)

Reply optimizer-training rows: 296
Reply calibration rows: 73
                   precision    recall  f1-score   support

       interested       0.62      0.42      0.50        12
meeting_requested       1.00      0.40      0.57        10
 more_information       0.56      0.50      0.53        10
         referral       0.56      0.83      0.67        12
          not_now       0.55      1.00      0.71        11
        objection       0.40      0.62      0.48        13
     wrong_person       0.40      0.33      0.36        12
    out_of_office       0.77      0.91      0.83        11
   not_interested       0.42      0.45      0.43        11
          opt_out       1.00      0.40      0.57        10
        ambiguous       0.83      0.42      0.56        12

         accuracy                           0.57       124
        macro avg       0.65      0.57      0.57       124
     weighted avg       0.64      0.57      0.56       124

TF-IDF validation macro-F1: 0.5652430331292474


Unfine-tuned reply baseline: {'accuracy': 0.13306451612903225, 'ece': 0.16774805512038932, 'loss': 1.5470689388532792, 'record_accuracy': 0.0}


## Training

In [11]:
_train_start = time.perf_counter()

In [12]:
reply_trainer = DecisionTrainer(
    model=reply_model,
    processing_class=reply_tokenizer,
    train_dataset=reply_train_items,
    eval_dataset=reply_validation_items,
    head_learning_rate=1e-4,
    args=TrainingArguments(
        per_device_train_batch_size=8,
        gradient_accumulation_steps=4,
        warmup_steps=10,
        num_train_epochs=2,
        learning_rate=2e-4,
        lr_scheduler_type="cosine",
        weight_decay=0.01,
        bf16=is_bfloat16_supported(),
        fp16=not is_bfloat16_supported(),
        eval_strategy="epoch",
        logging_steps=10,
        output_dir="outputs/reply-decision",
        report_to="none",
        seed=3407,
    ),
)

reply_training_stats = reply_trainer.train()
print(reply_training_stats.metrics)

Unsloth: Will smartly offload gradients to save VRAM!


Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Epoch,Training Loss,Validation Loss
1,1.452233,1.353936
2,0.794390,0.678314


Unsloth: Restored added_tokens_decoder metadata in /home/dev/work/lead-decision-models/outputs/reply-decision/checkpoint-20/.unsloth-save-sguqbd06/tokenizer_config.json.


{'train_runtime': 239.9684, 'train_samples_per_second': 2.467, 'train_steps_per_second': 0.083, 'total_flos': 0.0, 'train_loss': 1.123311710357666, 'epoch': 2.0}


In [13]:
TIMINGS["reply_training"] = round(time.perf_counter() - _train_start, 1)
peak_vram_gb = round(torch.cuda.max_memory_allocated() / 1024**3, 2)
print("Training seconds:", TIMINGS["reply_training"], "| peak VRAM GB:", peak_vram_gb)

Training seconds: 255.7 | peak VRAM GB: 2.64


## Calibration and evaluation

In [14]:
reply_calibration_metrics = FastDecisionModel.calibrate(
    reply_model,
    reply_tokenizer,
    reply_calibration_items,
)

print(reply_calibration_metrics)

{'accuracy': 0.8082191780821918, 'ece': 0.149264102112757, 'loss': 0.53029266798435, 'record_accuracy': 0.6301369863013698, 'fitted_types': [0, 2]}


In [15]:
reply_validation_metrics = FastDecisionModel.evaluate(
    reply_model,
    reply_tokenizer,
    reply_validation_items,
)

print(reply_validation_metrics)

{'accuracy': 0.7580645161290323, 'ece': 0.14793928832777084, 'loss': 0.6858404735478782, 'record_accuracy': 0.5887096774193549}


In [16]:
_predict_start = time.perf_counter()

In [17]:
from sklearn.metrics import confusion_matrix, f1_score

def predict_replies(frame, decision_model, decision_tokenizer):
    predictions = []

    for row in frame.to_dict(orient="records"):
        example = make_reply_record(row)

        result = FastDecisionModel.predict(
            decision_model,
            decision_tokenizer,
            example["state"],
            REPLY_QUESTIONS,
        )

        predictions.append(
            {
                "reply_intent": result["reply_intent"]["answer"],
                "follow_up_recommended": result["follow_up_recommended"]["answer"],
            }
        )

    return pd.DataFrame(predictions)

FastDecisionModel.for_inference(reply_model)

reply_validation_predictions = predict_replies(
    reply_validation_df,
    reply_model,
    reply_tokenizer,
)

print(
    classification_report(
        reply_validation_df["reply_intent"],
        reply_validation_predictions["reply_intent"],
        labels=REPLY_INTENTS,
        zero_division=0,
    )
)

validation_follow_up_labels = reply_validation_df["reply_intent"].map(
    lambda intent: intent not in NO_FOLLOW_UP_INTENTS
)

print(
    "Follow-up decision metrics:",
    classification_report(
        validation_follow_up_labels,
        reply_validation_predictions["follow_up_recommended"],
        labels=[False, True],
        zero_division=0,
    ),
)

print(
    "Macro-F1:",
    f1_score(
        reply_validation_df["reply_intent"],
        reply_validation_predictions["reply_intent"],
        labels=REPLY_INTENTS,
        average="macro",
        zero_division=0,
    ),
)

print(
    confusion_matrix(
        reply_validation_df["reply_intent"],
        reply_validation_predictions["reply_intent"],
        labels=REPLY_INTENTS,
    )
)

                   precision    recall  f1-score   support

       interested       0.60      0.25      0.35        12
meeting_requested       0.91      1.00      0.95        10
 more_information       1.00      0.60      0.75        10
         referral       0.80      1.00      0.89        12
          not_now       1.00      0.55      0.71        11
        objection       0.73      0.85      0.79        13
     wrong_person       1.00      0.33      0.50        12
    out_of_office       0.79      1.00      0.88        11
   not_interested       0.47      0.73      0.57        11
          opt_out       1.00      0.40      0.57        10
        ambiguous       0.37      0.83      0.51        12

         accuracy                           0.69       124
        macro avg       0.79      0.69      0.68       124
     weighted avg       0.78      0.69      0.68       124

Follow-up decision metrics:               precision    recall  f1-score   support

       False       0.00      

In [18]:
mean_predict_ms = round((time.perf_counter() - _predict_start) * 1000 / len(reply_validation_df), 1)
print("Mean milliseconds per reply prediction:", mean_predict_ms)

Mean milliseconds per reply prediction: 106.8


In [19]:
reply_test_metrics = FastDecisionModel.evaluate(
    reply_model,
    reply_tokenizer,
    reply_test_items,
)

FastDecisionModel.for_inference(reply_model)

reply_test_predictions = predict_replies(
    reply_test_df,
    reply_model,
    reply_tokenizer,
)

print("Unsloth reply test metrics:", reply_test_metrics)

print(
    classification_report(
        reply_test_df["reply_intent"],
        reply_test_predictions["reply_intent"],
        labels=REPLY_INTENTS,
        zero_division=0,
    )
)

reply_test_text_predictions = reply_text_baseline.predict(
    reply_test_df["reply_text"]
)

print(
    "TF-IDF test macro-F1:",
    f1_score(
        reply_test_df["reply_intent"],
        reply_test_text_predictions,
        labels=REPLY_INTENTS,
        average="macro",
        zero_division=0,
    ),
)

print(
    classification_report(
        reply_test_df["reply_intent"],
        reply_test_text_predictions,
        labels=REPLY_INTENTS,
        zero_division=0,
    )
)

test_follow_up_labels = reply_test_df["reply_intent"].map(
    lambda intent: intent not in NO_FOLLOW_UP_INTENTS
)

print(
    "Follow-up test metrics:",
    classification_report(
        test_follow_up_labels,
        reply_test_predictions["follow_up_recommended"],
        labels=[False, True],
        zero_division=0,
    ),
)

print(
    "Test macro-F1:",
    f1_score(
        reply_test_df["reply_intent"],
        reply_test_predictions["reply_intent"],
        labels=REPLY_INTENTS,
        average="macro",
        zero_division=0,
    ),
)

Unsloth reply test metrics: {'accuracy': 0.7943548387096774, 'ece': 0.160946637752556, 'loss': 0.5606553069345893, 'record_accuracy': 0.6209677419354839}
                   precision    recall  f1-score   support

       interested       1.00      0.54      0.70        13
meeting_requested       0.85      1.00      0.92        11
 more_information       1.00      0.70      0.82        10
         referral       1.00      0.62      0.76        13
          not_now       0.70      0.70      0.70        10
        objection       0.64      0.75      0.69        12
     wrong_person       0.90      0.75      0.82        12
    out_of_office       0.91      1.00      0.95        10
   not_interested       0.71      0.91      0.80        11
          opt_out       1.00      0.70      0.82        10
        ambiguous       0.43      0.83      0.57        12

         accuracy                           0.77       124
        macro avg       0.83      0.77      0.78       124
     weighted avg 

## Routing policy

In [20]:
REPLY_ACTIONS = {
    "interested": "handoff_to_sales",
    "meeting_requested": "handoff_to_sales",
    "more_information": "prepare_requested_information",
    "referral": "review_referral",
    "not_now": "schedule_later_after_date_confirmation",
    "objection": "human_review",
    "wrong_person": "human_review",
    "out_of_office": "pause_until_return_date_confirmation",
    "not_interested": "stop_sequence",
    "opt_out": "suppress_marketing",
    "ambiguous": "human_review",
}

def route_reply(
    predicted_intent,
    model_follow_up_recommended,
    contact_is_suppressed,
    explicit_opt_out_event,
):
    if isinstance(contact_is_suppressed, np.bool_):
        contact_is_suppressed = bool(contact_is_suppressed)
    if isinstance(explicit_opt_out_event, np.bool_):
        explicit_opt_out_event = bool(explicit_opt_out_event)

    if (
        contact_is_suppressed is True
        or explicit_opt_out_event is True
        or predicted_intent == "opt_out"
    ):
        return {
            "action": "suppress_marketing",
            "model_follow_up_recommended": (
                model_follow_up_recommended
                if isinstance(model_follow_up_recommended, (bool, np.bool_))
                else None
            ),
            "policy_follow_up_recommended": False,
            "send": False,
        }

    if (
        contact_is_suppressed is not False
        or explicit_opt_out_event is not False
    ):
        return {
            "action": "verify_suppression_and_opt_out_status",
            "model_follow_up_recommended": None,
            "policy_follow_up_recommended": False,
            "send": False,
        }

    if not isinstance(model_follow_up_recommended, (bool, np.bool_)):
        return {
            "action": "human_review",
            "reason": "invalid_follow_up_model_output",
            "model_follow_up_recommended": None,
            "policy_follow_up_recommended": False,
            "send": False,
        }

    model_follow_up_recommended = bool(model_follow_up_recommended)

    if predicted_intent not in REPLY_ACTIONS:
        return {
            "action": "human_review",
            "reason": "unknown_reply_intent",
            "model_follow_up_recommended": model_follow_up_recommended,
            "policy_follow_up_recommended": False,
            "send": False,
        }

    policy_follow_up_recommended = (
        predicted_intent not in NO_FOLLOW_UP_INTENTS
    )

    if not policy_follow_up_recommended:
        return {
            "action": REPLY_ACTIONS[predicted_intent],
            "model_follow_up_recommended": model_follow_up_recommended,
            "policy_follow_up_recommended": False,
            "send": False,
        }

    if not model_follow_up_recommended:
        return {
            "action": "human_review",
            "reason": "intent_and_follow_up_decisions_disagree",
            "model_follow_up_recommended": False,
            "policy_follow_up_recommended": True,
            "send": False,
        }

    return {
        "action": REPLY_ACTIONS[predicted_intent],
        "model_follow_up_recommended": True,
        "policy_follow_up_recommended": True,
        "send": False,
    }

In [21]:
sample_reply = reply_validation_predictions.iloc[0]

print(
    route_reply(
        predicted_intent=sample_reply["reply_intent"],
        model_follow_up_recommended=sample_reply["follow_up_recommended"],
        contact_is_suppressed=False,
        explicit_opt_out_event=False,
    )
)

{'action': 'schedule_later_after_date_confirmation', 'model_follow_up_recommended': True, 'policy_follow_up_recommended': True, 'send': False}


In [22]:
import numpy as np

assert route_reply(
    predicted_intent="interested",
    model_follow_up_recommended=True,
    contact_is_suppressed=False,
    explicit_opt_out_event=True,
)["action"] == "suppress_marketing"

assert route_reply(
    predicted_intent="interested",
    model_follow_up_recommended=True,
    contact_is_suppressed=None,
    explicit_opt_out_event=False,
)["action"] == "verify_suppression_and_opt_out_status"

assert route_reply(
    predicted_intent="interested",
    model_follow_up_recommended=False,
    contact_is_suppressed=False,
    explicit_opt_out_event=False,
)["action"] == "human_review"

assert route_reply(
    predicted_intent="interested",
    model_follow_up_recommended="False",
    contact_is_suppressed=False,
    explicit_opt_out_event=False,
)["reason"] == "invalid_follow_up_model_output"

assert route_reply(
    predicted_intent="opt_out",
    model_follow_up_recommended="invalid",
    contact_is_suppressed=False,
    explicit_opt_out_event=False,
)["action"] == "suppress_marketing"

assert route_reply(
    predicted_intent="interested",
    model_follow_up_recommended="invalid",
    contact_is_suppressed=np.bool_(True),
    explicit_opt_out_event=np.bool_(False),
)["action"] == "suppress_marketing"

## Save, publish and release

In [23]:
reply_model.save_pretrained("outputs/reply-decision-adapter")
reply_model.save_pretrained_merged(
    "outputs/reply-decision-merged",
    reply_tokenizer,
)

Unsloth: Restored added_tokens_decoder metadata in /home/dev/work/lead-decision-models/outputs/reply-decision-adapter/.unsloth-save-4383jpvw/tokenizer_config.json.


Unsloth: Restored added_tokens_decoder metadata in /home/dev/work/lead-decision-models/outputs/reply-decision-merged/.unsloth-save-3ngzbq4l/tokenizer_config.json.


/home/dev/work/lead-decision-models/unsloth_env/lib/python3.13/site-packages/unsloth_zoo/saving_utils.py:5436: UserWarning: Unsloth: Could not read the base config from `LiquidAI/LFM3-1.7B` (The repository LiquidAI/LFM3-1.7B contains custom code which must be executed to correctly load the model. You can inspect the repository content at https://hf.co/LiquidAI/LFM3-1.7B .
 You can inspect the repository content at https://hf.co/LiquidAI/LFM3-1.7B.
Please pass the argument `trust_remote_code=True` to allow custom code to be run.). Falling back to the in-memory config, which might not describe the exported weights (see #969).
  warnings.warn(


/home/dev/work/lead-decision-models/unsloth_env/lib/python3.13/site-packages/unsloth_zoo/saving_utils.py:3849: UserWarning: Unsloth: `LiquidAI/LFM3-1.7B` was loaded without a recorded commit, so its repo code was not copied into the export. Copy the repo's *.py files at the revision you loaded into `/home/dev/work/lead-decision-models/outputs/reply-decision-merged/.unsloth-save-3ngzbq4l` before loading it with trust_remote_code=True.
  warnings.warn(


Found HuggingFace hub cache directory: /home/dev/.cache/huggingface/hub
Checking cache directory for required files...


Unsloth: Copying 1 files from cache to `/home/dev/work/lead-decision-models/outputs/reply-decision-merged/.unsloth-save-3ngzbq4l`:   0%|          | 0/1 [00:00<?, ?it/s]

... (5 outputs left out)

Unsloth: Merging weights into 16bit:   0%|          | 0/1 [00:00<?, ?it/s]

Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [00:02<00:00,  2.62s/it]

Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [00:02<00:00,  2.62s/it]

Unsloth: Merge process complete. Saved to `/home/dev/work/lead-decision-models/outputs/reply-decision-merged/.unsloth-save-3ngzbq4l`


In [24]:
REPLY_REPO = f"{HF_USERNAME}/lead-decision-models-reply-router"

if PUBLISH:
    reply_model.push_to_hub(REPLY_REPO, reply_tokenizer)
    reply_model.push_to_hub_merged(f"{REPLY_REPO}-merged", reply_tokenizer, save_method="merged_16bit")
    print("Published", REPLY_REPO, "and", f"{REPLY_REPO}-merged")

Unsloth: Restored added_tokens_decoder metadata in /tmp/tmp86pw1e6g/.unsloth-save-pq004qpp/tokenizer_config.json.


Unsloth: Saved the decision model to https://huggingface.co/raoofaltaher/lead-decision-models-reply-router


Unsloth: Restored added_tokens_decoder metadata in /tmp/tmp8ylivean/.unsloth-save-nag4hgij/tokenizer_config.json.


/home/dev/work/lead-decision-models/unsloth_env/lib/python3.13/site-packages/unsloth_zoo/saving_utils.py:3849: UserWarning: Unsloth: `LiquidAI/LFM3-1.7B` was loaded without a recorded commit, so its repo code was not copied into the export. Copy the repo's *.py files at the revision you loaded into `/tmp/tmp8ylivean/.unsloth-save-nag4hgij` before loading it with trust_remote_code=True.
  warnings.warn(


Found HuggingFace hub cache directory: /home/dev/.cache/huggingface/hub
Checking cache directory for required files...


Unsloth: Copying 1 files from cache to `/tmp/tmp8ylivean/.unsloth-save-nag4hgij`:   0%|          | 0/1 [00:00<?, ?it/s]

... (7 outputs left out)

Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [00:03<00:00,  3.73s/it]

Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [00:03<00:00,  3.73s/it]

Unsloth: Merge process complete. Saved to `/tmp/tmp8ylivean/.unsloth-save-nag4hgij`


Unsloth: Saved the decision model to https://huggingface.co/raoofaltaher/lead-decision-models-reply-router-merged
Published raoofaltaher/lead-decision-models-reply-router and raoofaltaher/lead-decision-models-reply-router-merged


In [25]:
import gc

del reply_trainer, reply_model, reply_tokenizer, reply_training_stats
gc.collect()
torch.cuda.empty_cache()

## Reload the merged model

In [26]:
reloaded_reply_model, reloaded_reply_tokenizer = FastDecisionModel.from_pretrained(
    model_name="outputs/reply-decision-merged",
    trust_remote_code=True,
    max_seq_length=2048,
    load_in_4bit=LOAD_IN_4BIT,
)

FastDecisionModel.for_inference(reloaded_reply_model)

reply_example = make_reply_record(reply_test_df.iloc[0])
reply_reload_prediction = FastDecisionModel.predict(
    reloaded_reply_model,
    reloaded_reply_tokenizer,
    reply_example["state"],
    REPLY_QUESTIONS,
)

print(reply_reload_prediction)

reply_before_export = reply_test_predictions.iloc[0]
print(
    "Reply class agreement after export:",
    reply_reload_prediction["reply_intent"]["answer"]
    == reply_before_export["reply_intent"],
    reply_reload_prediction["follow_up_recommended"]["answer"]
    == bool(reply_before_export["follow_up_recommended"]),
)

Unsloth: WARNING `trust_remote_code` is True.
Are you certain you want to do remote code execution?
==((====))==  Unsloth 2026.10.11: Fast Lfm3 patching. Transformers: 5.22.1.
   \\   /|    NVIDIA GeForce RTX 5090 Laptop GPU. Num GPUs = 1. Max memory: 23.863 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.14.1+cu132. CUDA: 12.0. CUDA Toolkit: 13.2. Triton: 3.8.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/286 [00:00<?, ?it/s]

Unsloth: Repaired remote modeling code so it trains: Lfm3ForCausalLM.forward.


Unsloth: compiling each transformer block as one region (Lfm3DecoderLayer).


{'reply_intent': {'type': 'choice', 'choice': 'interested', 'confidence': 0.6755, 'probabilities': {'interested': 0.6755, 'meeting_requested': 0.0159, 'more_information': 0.0144, 'referral': 0.0155, 'not_now': 0.0118, 'objection': 0.0134, 'wrong_person': 0.0164, 'out_of_office': 0.012, 'not_interested': 0.0119, 'opt_out': 0.0115, 'ambiguous': 0.2017}, 'answer': 'interested'}, 'follow_up_recommended': {'type': 'noul', 'noul': 0.8179, 'answer': True, 'probabilities': {'true': 0.8179, 'false': 0.1821}}}
Reply class agreement after export: True True


## Metrics and experiment manifest

In [27]:
import hashlib
import importlib.metadata
import platform

def installed_version(distribution):
    try:
        return importlib.metadata.version(distribution)
    except importlib.metadata.PackageNotFoundError:
        return None

reply_metrics = {
    "model_id": MODEL_ID,
    "model_revision": MODEL_REVISION,
    "load_in_4bit": LOAD_IN_4BIT,
    "splits": {
        "optimizer_training": len(reply_fit_df),
        "calibration": len(reply_calibration_df),
        "validation": len(reply_validation_df),
        "test": len(reply_test_df),
    },
    "baseline_untrained": reply_baseline,
    "tfidf": {
        "validation_macro_f1": float(f1_score(reply_validation_df["reply_intent"], reply_validation_text_predictions, labels=REPLY_INTENTS, average="macro", zero_division=0)),
        "test_macro_f1": float(f1_score(reply_test_df["reply_intent"], reply_test_text_predictions, labels=REPLY_INTENTS, average="macro", zero_division=0)),
    },
    "calibration": reply_calibration_metrics,
    "validation": {**reply_validation_metrics, "macro_f1": float(f1_score(reply_validation_df["reply_intent"], reply_validation_predictions["reply_intent"], labels=REPLY_INTENTS, average="macro", zero_division=0))},
    "test": {**reply_test_metrics, "macro_f1": float(f1_score(reply_test_df["reply_intent"], reply_test_predictions["reply_intent"], labels=REPLY_INTENTS, average="macro", zero_division=0))},
    "test_report": classification_report(reply_test_df["reply_intent"], reply_test_predictions["reply_intent"], labels=REPLY_INTENTS, output_dict=True, zero_division=0),
    "follow_up_report": classification_report(test_follow_up_labels, reply_test_predictions["follow_up_recommended"], labels=[False, True], output_dict=True, zero_division=0),
    "confusion_matrix": {"labels": REPLY_INTENTS, "matrix": confusion_matrix(reply_test_df["reply_intent"], reply_test_predictions["reply_intent"], labels=REPLY_INTENTS).tolist()},
    "timings_seconds": TIMINGS,
    "peak_vram_gb": peak_vram_gb,
    "mean_predict_ms": mean_predict_ms,
}
Path("outputs/reply-metrics.json").write_text(json.dumps(reply_metrics, indent=2, default=float), encoding="utf-8")

lead_metrics = json.loads(Path("outputs/lead-metrics.json").read_text(encoding="utf-8"))

experiment_manifest = {
    "experiment": "unsloth-b2b-lead-and-reply-decision-models",
    "dataset": {
        "leadforge_commit": lead_metrics["dataset"]["leadforge_commit"],
        "leadforge_release": "v0.1.0-alpha",
        "leadforge_tier": "intermediate",
        "leadforge_task": "converted_within_90_days",
        "lead_target_counts": lead_metrics["dataset"]["target_counts"],
        "reply_dataset": "synthetic-llm-generated-automatically-checked",
        "reply_dataset_sha256": reply_dataset_hash,
        "reply_intent_counts": {key: int(value) for key, value in reply_df["reply_intent"].value_counts().items()},
    },
    "base_model": {"id": MODEL_ID, "revision": MODEL_REVISION, "load_in_4bit": LOAD_IN_4BIT},
    "splits": {"lead": lead_metrics["splits"], "reply": reply_metrics["splits"]},
    "seed": 3407,
    "calibration_split_seed": 3408,
    "lora": {"rank": 16, "alpha": 16, "dropout": 0.0},
    "training": {
        "learning_rate": 2e-4,
        "head_learning_rate": 1e-4,
        "epochs": 2,
        "per_device_train_batch_size": 8,
        "gradient_accumulation_steps": 4,
        "max_seq_length": 2048,
    },
    "timings_seconds": {**lead_metrics["timings_seconds"], **TIMINGS},
    "peak_vram_gb": {"lead": lead_metrics["peak_vram_gb"], "reply": peak_vram_gb},
    "runtime": {
        "python": platform.python_version(),
        "pytorch": torch.__version__,
        "cuda_runtime": torch.version.cuda,
        "gpu": torch.cuda.get_device_name(0),
        "packages": {
            name: installed_version(name)
            for name in ["unsloth", "unsloth_zoo", "transformers", "bitsandbytes", "datasets", "pandas", "scikit-learn", "huggingface-hub"]
        },
    },
    "policy_versions": {
        "icp_policy": "fictional-procurement-icp-v1",
        "reply_taxonomy": "b2b-reply-intents-v1",
        "outreach_policy": "human-approval-and-external-suppression-v1",
    },
}

Path("outputs/experiment-manifest.json").write_text(json.dumps(experiment_manifest, indent=2, default=float), encoding="utf-8")
print("Wrote outputs/reply-metrics.json and outputs/experiment-manifest.json")

Wrote outputs/reply-metrics.json and outputs/experiment-manifest.json
